# Coefficient Functions

Rendering NGSolve `CoefficientFunction`s on meshes using `ngsolve_webgpu`.

- `FunctionData` stores the GPU representation of a CoefficientFunction.
- `CFRenderer` renders CoefficientFunctions on 2D surface elements.
- `ClippingCF` adds a cross-section view for 3D meshes.
- For vector-valued CFs, a component dropdown appears automatically.

## Rendering a Scalar Function on a Surface

Create a `MeshData` from a mesh, wrap a CF in `FunctionData`, and render with `CFRenderer`. Add a `Colorbar` for reference.

In [ ]:
from netgen.occ import *
from ngsolve import *
from ngsolve_webgpu import *
from webgpu.jupyter import Draw

geo = OCCGeometry(Cylinder((0, 0, 0), X, r=0.3, h=0.5))
mesh = Mesh(geo.GenerateMesh(maxh=6))
mesh.Curve(5)

mesh_data = MeshData(mesh)
function_data = FunctionData(mesh_data, sin(20 * x), order=5)
cfrenderer = CFRenderer(function_data)
cfrenderer.colormap.set_min_max(-1, 1)
Draw([cfrenderer, Colorbar(cfrenderer.colormap)])

## Multi-Component Functions

For vector-valued CFs on 3D meshes, use `ClippingCF` to visualize a cross-section. Connect `CFRenderer` and `ClippingCF` via a shared colormap and clipping plane. The component selector lets you switch between vector components.

In [ ]:
from ngsolve import *
from ngsolve_webgpu import *
from webgpu.clipping import Clipping
from webgpu.jupyter import Draw

mesh = Mesh(unit_cube.GenerateMesh(maxh=0.1))
cf = CF((sin(10 * z) * cos(15 * x), (x-0.5)**2 + (y-0.5)**2 + (z-0.5)**2 - 2))

mesh_data = MeshData(mesh)
function_data = FunctionData(mesh_data, cf, order=5)

clipping = Clipping()
clipping.mode = clipping.Mode.PLANE
clipping.center = [0.7, 0.5, 0.5]
clipping.normal = [1, -1, 1]

colormap = Colormap()
clip = ClippingCF(function_data, clipping=clipping, colormap=colormap)
cfr = CFRenderer(function_data, colormap=colormap, clipping=clipping)
cfr.on_component_change(clip.set_component)

scene = Draw([cfr, clip, Colorbar(colormap)])
clipping.add_options_to_gui(scene.gui)
cfr.add_options_to_gui(scene.gui)